In [ ]:
forget()
load("Library.sage")

R.<x,y,z,w> = LaurentPolynomialRing(QQ);
R_poly = R.polynomial_ring();
Q.<t> = PolynomialRing(QQ);

load("QuiverFlagZeroLoci_original.sage")
load("QuiverFlagZeroLoci_mutated.sage")


## Print period sequences

def test_period_sequences():
    print("Period sequences:");
    for LG in F_list:
        print(F_list_names[F_list.index(LG)])
        print(period_sequence(LG, 10));


## Check the binomial property

def test_edge_polynomials():
    for LG in F_list:
        polytope = newton_polytope(LG);
        polyhedron = Polyhedron(polytope.vertices(), base_ring=ZZ);

        integral_points = polyhedron.integral_points();
        interior_points = polytope.interior_points();

        edges = polytope.edges();
        faces = polytope.faces(2);
        facets = polytope.facets();
    
        edge_poly_list = [];
        failed_edges = [];
        for edge in edges:
            edge_polyhedron = Polyhedron(edge.vertices(), base_ring=ZZ);
            edge_polynomial = 0;
            integral_points_number = len(edge_polyhedron.integral_points());
            for i in range(integral_points_number):
                point = edge_polyhedron.integral_points()[i];
                monomial = R.monomial(*list(point));
                coeff = LG.monomial_coefficient(monomial);
                edge_polynomial += coeff*t^i;
            edge_poly_list.append(factor(edge_polynomial));
            if (edge_polynomial != (1 + t)^edge_polynomial.degree()):
                failed_edges.append(edge);
        if (failed_edges == []) :
            print("OK: " + str(F_list_names[F_list.index(LG)]));
        else:
            print("FAIL: " + str(F_list_names[F_list.index(LG)]));


## Check that all irreducible 2d Minkowski summands
## are hollow triangles of lattice width 1

def test_2d_minkowski_summands():
    for LG in F_list:
        bad = 0;
        for [P,Q,S] in face_minkowski_polytopes(LG, 2, refined = False):
            if (P.dim() != 2) : continue;
            if (len(P.vertices()) != 3) :
                bad = 1; break;
            if (len(P.interior_points()) > 0) :
                bad = 1; break;
            # We want to exclude the hollow triangle of lattice width 2
            if (P.polyhedron().volume() == 2) :
                if (max(P.facet_constants()) != 4):
                    bad = 1; break;
        if not (bad):
            print("OK: " + str(F_list_names[F_list.index(LG)]));
        else:
            print("FAIL: " + str(F_list_names[F_list.index(LG)]));


## For all facet polynomial print generators of ideals
## of intersections of its irreducible components
## while omitting linear generators and cases when
## there is only one non-linear generator
## (i.e., the only cases when the rationality
## over Q of these intersections is not immediate)

def test_components_rationality():
    for LG in F_list:
        bad = 0;
        for List in face_minkowski_polytopes(LG, 3, refined = True):
            L = len(List);
            par = List[0][-2].parent();
            par_poly = par.polynomial_ring();
            frac_field = FractionField(par_poly);
            T.<X_0, X_1, X_2> = toric_varieties.torus(3);

            for I in Subsets(range(L)):
                if (len(I) < 2) : continue;
                gens_list = [];
                component_list = [];

                for i in list(I): gens_list.append((List[i])[-2]);
                P = T.subscheme(par.ideal(gens_list).radical());
                
                # Check that irreducible components of the intersection are rational over Q
                for PP in P.irreducible_components():
                    component_list.append(PP.defining_polynomials());
                    II = par.ideal(PP.defining_polynomials()).radical();
                    reduced_list = [];
                    for poly in II.gens():
                        poly_reduced = par_poly(frac_field(poly).numerator());
                        if (poly_reduced.degree() != 1) : reduced_list.append(poly);
                    if (len(reduced_list) > 1): print(reduced_list);

                # Check that the irreducible component of the intersection do not intersect
                for J in Subsets(range(len(component_list))):
                    if (len(J) < 2) : continue;
                    print("Warning: we actually have a reducible intersection.");
                    intersection_list = [];
                    for j in list(J): intersection_list += component_list[j];
                    intersection_ideal = par.ideal(intersection_list);
                    if (intersection_ideal.radical().gens().count(1) == 0):
                        print("Warning: components of the intersection are not disjoint.");


## Present facet polynomials in the form
## F(X_0, X_1) * X_2 + G(X_0, X_1) = 0, and check that components of
## F(X_0, X_1) = G(X_0, X_1) = 0 are defined over Q.

def test_boundary_rationality():
    for LG in F_list:
        for [P,Q,R] in face_minkowski_polytopes(LG, 3, refined = False):
            par = Q.parent();
            par_poly = par.polynomial_ring();
            frac_field = FractionField(par_poly);
            poly_reduced = par_poly(frac_field(Q).numerator());
        
            if (newton_polytope(Q).dim() != 3) : continue;
            output = linearize_width_one(Q)
            F = output['F']; G = output['G'];
            F = par_poly(frac_field(F).numerator());
            G = par_poly(frac_field(G).numerator());

            are_all_generators_linear = 1;
                       
            for J in par_poly.ideal([F,G]).radical().primary_decomposition():
                are_generators_linear = 1;
                for gen in J.gens():
                    if (gen.degree() != 1) :
                        are_generators_linear = 0;
                        break;
                if not (are_generators_linear):
                    print(J.gens());
                    are_all_generators_linear = 0;

            if not (are_all_generators_linear) :
                print("Warning! Not all generators are linear.");
            else:
                print("OK!");


## Does F_list_mutated actually provides fixed Laurent polynomials 
## (period matching, transverse non-degeneracy, reflexive Newton polytope)

def test_replacement_nondegeneracy():
    for LG_fixed in F_list_mutated:
        i = F_list_mutated.index(LG_fixed);
        LG_orig = F_sublist_transverse_ND_fails[i];
        j = F_list.index(LG_orig);   
        LG_name = F_list_names[j];
        period_bool = (period_sequence(LG_orig, 5) == period_sequence(LG_fixed, 5));
        nondegeneracy_bool = (is_transverse_nondegenerate(LG_fixed));
        newton_bool = Polyhedron(newton_polytope(LG_fixed).vertices()).is_reflexive();
        print([LG_name, period_bool, nondegeneracy_bool, newton_bool]);


## Print facet polynomials of a specific LG model
## after all possible coordinate changes

#face_polynomial(F_566, 3)


## Check the transverse non-degeneracy property

#for LG in F_list:
#    if not (is_transverse_nondegenerate(LG) == True):
#        print(F_list_names[F_list.index(LG)])


## Tests

#test_period_sequences()
#test_edge_polynomials()
#test_2d_minkowski_summands()
#test_components_rationality()
#test_boundary_rationality()
#test_replacement_nondegeneracy()